# PySTEPS vs Persistence Baseline

This notebook evaluates the performance of the simple persistence baseline against the optical flow-based `pysteps` model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jalrakshak_ml.nowcast.persistence import PersistenceNowcast
from jalrakshak_ml.nowcast.pysteps_adapter import PystepsNowcast
from jalrakshak_ml.evaluation.evaluator import NowcastEvaluator

%matplotlib inline

## Generate Synthetic Data
We will create a simple moving rainfall blob to test the optical flow algorithm.

In [ ]:
def create_synthetic_rainfall(frames=10, size=256):
    """Creates a moving rainfall blob."""
    data = np.zeros((frames, size, size))
    
    # Create a 2D Gaussian blob
    x, y = np.meshgrid(np.arange(size), np.arange(size))
    
    start_x, start_y = 50, 50
    speed_x, speed_y = 10, 10 # Pixels per frame
    
    for i in range(frames):
        cx = start_x + i * speed_x
        cy = start_y + i * speed_y
        blob = 15.0 * np.exp(-((x - cx)**2 + (y - cy)**2) / 400.0)
        data[i] = blob
        
    return data

rainfall_seq = create_synthetic_rainfall(frames=6)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i in range(6):
    axes[i].imshow(rainfall_seq[i], vmin=0, vmax=15, cmap='Blues')
    axes[i].set_title(f't={i}')
    axes[i].axis('off')
plt.tight_layout()

## Generate Nowcasts
We will use the first 3 frames (t=0, 1, 2) as history to predict the next 3 frames (t=3, 4, 5).

In [ ]:
history = rainfall_seq[:3]
ground_truth = rainfall_seq[3:]
lead_times = 3

persist = PersistenceNowcast()
pred_persist = persist.predict(history[-1], lead_times=lead_times)

pysteps_model = PystepsNowcast()
pred_pysteps = pysteps_model.predict(history, lead_times=lead_times)

## Visualization

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for i in range(lead_times):
    axes[0, i].imshow(ground_truth[i], vmin=0, vmax=15, cmap='Blues')
    axes[0, i].set_title(f'Ground Truth (t+{i+1})')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(pred_persist[i], vmin=0, vmax=15, cmap='Blues')
    axes[1, i].set_title(f'Persistence (t+{i+1})')
    axes[1, i].axis('off')
    
    axes[2, i].imshow(pred_pysteps[i], vmin=0, vmax=15, cmap='Blues')
    axes[2, i].set_title(f'PySTEPS (t+{i+1})')
    axes[2, i].axis('off')
    
plt.tight_layout()

## Quantitative Evaluation

In [ ]:
evaluator = NowcastEvaluator(thresholds=[1.0, 5.0])

df_persist = evaluator.evaluate_sequence(ground_truth, pred_persist)
df_pysteps = evaluator.evaluate_sequence(ground_truth, pred_pysteps)

df_persist['model'] = 'Persistence'
df_pysteps['model'] = 'PySTEPS'

import pandas as pd
df_all = pd.concat([df_persist, df_pysteps])
df_all